# Taller 2: Limpieza de Datos

**Gestión de Datos · Módulo 1 · 2026** — MSc. Edison Leonardo Neira Espitia

**Integrantes:** Luis Alberto Banda, Maria Camila Rincón, Juan Pablo Marin, Gabriel Camilo Velasquez y Juan Camilo Guerrero

**Entorno:** Docker con Jupyter Lab (imagen `quay.io/jupyter/scipy-notebook`); ver `Dockerfile`, `docker-compose.yml` y `requirements.txt`.

**Dataset:** Titanic — https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

El desarrollo también se encuentra en el README (Revisarlo).

## Ejercicio 1: Diagnóstico de calidad de los datos

### 4.1 Carga y vistazo inicial

Se importan las librerías y se carga `titanic.csv`. La primera vez se descarga desde la URL del enunciado y se guarda una copia local en `work/titanic.csv`, para que las siguientes ejecuciones no dependan de internet.

In [1]:
# Librerías del taller
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Mostrar todas las columnas al imprimir DataFrames
pd.set_option("display.max_columns", None)

In [2]:
# Fuente del dataset y copia local
URL_TITANIC = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
RUTA_LOCAL = Path("titanic.csv")

# Descargar solo si no existe la copia local
if not RUTA_LOCAL.exists():
    pd.read_csv(URL_TITANIC).to_csv(RUTA_LOCAL, index=False)

df = pd.read_csv(RUTA_LOCAL)

# 10 primeras filas
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [3]:
# Dimensiones del dataset: (filas, columnas)
print("shape:", df.shape)

shape: (891, 12)


In [4]:
# Tipo de dato de cada columna
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

#### Reporte exploratorio con ydata-profiling

Se genera el reporte automático y se guarda en `work/titanic_profiling.html`. Sirve solo como apoyo: toda conclusión del diagnóstico se verifica con una celda propia más adelante.

Nota: `ydata_profiling` emite un `DeprecationWarning` al importarse (anuncia que el paquete cambiará de nombre). Se silencia solo ese aviso para que el notebook corra limpio; no afecta el reporte.

In [5]:
import warnings

# Silenciar únicamente el aviso de deprecación que emite ydata_profiling al importarse
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    from ydata_profiling import ProfileReport

# Reporte exploratorio del dataset original (sin limpiar)
perfil = ProfileReport(df, title="Titanic - Reporte exploratorio", explorative=True, progress_bar=False)

RUTA_REPORTE = Path("titanic_profiling.html")
perfil.to_file(RUTA_REPORTE)
print("Reporte guardado en:", RUTA_REPORTE.resolve())

100%|██████████| 12/12 [00:00<00:00, 136.22it/s]


Reporte guardado en: /home/jovyan/work/titanic_profiling.html


### 4.2 Tabla de diagnóstico por columna

Se construye una tabla con una fila por cada una de las 12 columnas y estas verificaciones:

- **Valores únicos o constantes:** columnas con un solo valor o con un valor distinto por fila (identificadores).
- **Valores faltantes:** número y % de nulos, y mecanismo probable (MCAR, MAR o MNAR).
- **Faltantes disfrazados:** ceros imposibles en `Age` y `Fare`, marcadores de texto (`"?"`, `"Unknown"`, `"NA"`, vacío, etc.) y tiquetes sin ningún dígito.
- **Valores atípicos por atributo:** límites de Tukey (Q1 − 1,5·IQR, Q3 + 1,5·IQR) para `Age`, `Fare`, `SibSp` y `Parch`.
- **Registros atípicos:** filas que combinan valores extremos en varios atributos, o con una tarifa atípica para su propia clase.
- **Redundancia o nulo valor:** correlación de Spearman (en valor absoluto) con la columna numérica más parecida, y comparación de `SibSp` y `Parch` frente al tamaño de familia.

La última columna de la tabla es la **acción de mitigación propuesta**. El mecanismo y la acción son un **borrador del agente** que el grupo debe revisar y corregir.

In [6]:
# Parámetros de las verificaciones
N_FILAS = len(df)
NUM_TUKEY = ["Age", "Fare", "SibSp", "Parch"]
COLS_CERO_IMPOSIBLE = ["Age", "Fare"]  # un 0 en edad o tarifa no es plausible
MARCADORES = {"?", "unknown", "na", "n/a", "none", "null", "-", ""}


def limites_tukey(serie):
    """Límites de Tukey: (Q1 - 1,5·IQR, Q3 + 1,5·IQR)."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


def mascara_tukey(serie):
    """True donde el valor cae fuera de los límites de Tukey."""
    li, ls = limites_tukey(serie)
    return (serie < li) | (serie > ls)


def contar_disfrazados(col):
    """Cuenta faltantes disfrazados según el tipo de columna."""
    s = df[col].dropna()
    if col in COLS_CERO_IMPOSIBLE:
        return int((s == 0).sum()), "valor 0"
    if s.dtype == object:
        marcador = s.str.strip().str.lower().isin(MARCADORES)
        if col == "Ticket":
            sin_digitos = ~s.str.contains(r"\d")
            return int((marcador | sin_digitos).sum()), "marcador o ticket sin dígitos"
        return int(marcador.sum()), "marcador"
    return 0, "no aplica"


diagnostico = []
for col in df.columns:
    n_unicos = df[col].nunique(dropna=True)
    if n_unicos <= 1:
        tipo_unico = "Constante"
    elif n_unicos == N_FILAS:
        tipo_unico = "Un valor por fila"
    else:
        tipo_unico = "—"
    n_nulos = int(df[col].isna().sum())
    n_disfr, criterio = contar_disfrazados(col)
    fila = {
        "Columna": col,
        "dtype": str(df[col].dtype),
        "N únicos": n_unicos,
        "Único/constante": tipo_unico,
        "Nulos": n_nulos,
        "% nulos": round(100 * n_nulos / N_FILAS, 2),
        "Disfrazados": n_disfr,
        "Criterio disfrazado": criterio,
    }
    if col in NUM_TUKEY:
        li, ls = limites_tukey(df[col])
        n_atip = int(mascara_tukey(df[col]).sum())
        fila.update({
            "Tukey inf": round(li, 2),
            "Tukey sup": round(ls, 2),
            "Atípicos": n_atip,
            "% atípicos": round(100 * n_atip / N_FILAS, 2),
        })
    diagnostico.append(fila)

diagnostico = pd.DataFrame(diagnostico).set_index("Columna")
diagnostico

,dtype,N únicos,Único/constante,Nulos,% nulos,Disfrazados,Criterio disfrazado,Tukey inf,Tukey sup,Atípicos,% atípicos
Columna,,,,,,,,,,,
PassengerId,int64,891,Un valor por fila,0,0.00,0,no aplica,NaN,NaN,NaN,NaN
Survived,int64,2,—,0,0.00,0,no aplica,NaN,NaN,NaN,NaN
Pclass,int64,3,—,0,0.00,0,no aplica,NaN,NaN,NaN,NaN
Name,object,891,Un valor por fila,0,0.00,0,marcador,NaN,NaN,NaN,NaN
Sex,object,2,—,0,0.00,0,marcador,NaN,NaN,NaN,NaN
Age,float64,88,—,177,19.87,0,valor 0,-6.69,64.81,11.0,1.23
SibSp,int64,7,—,0,0.00,0,no aplica,-1.50,2.50,46.0,5.16
Parch,int64,7,—,0,0.00,0,no aplica,0.00,0.00,213.0,23.91
Ticket,object,681,—,0,0.00,4,marcador o ticket sin dígitos,NaN,NaN,NaN,NaN


#### Mecanismo de los faltantes

Para proponer el mecanismo se compara el % de nulos de `Age` y `Cabin` entre clases (`Pclass`) y entre sobrevivientes y no sobrevivientes (`Survived`). Si el % cambia mucho entre grupos, el faltante depende de otra variable observada (MAR) y no es completamente al azar. También se revisan las filas sin `Embarked`.

In [7]:
# % de nulos de Age y Cabin según Pclass y según Survived
for grupo in ["Pclass", "Survived"]:
    pct_nulos = df[["Age", "Cabin"]].isna().groupby(df[grupo]).mean().mul(100).round(1)
    print(f"% de nulos por {grupo}:")
    print(pct_nulos, "\n")

# Filas sin puerto de embarque
df.loc[df["Embarked"].isna()]

% de nulos por Pclass:
         Age  Cabin
Pclass             
1       13.9   18.5
2        6.0   91.3
3       27.7   97.6 

% de nulos por Survived:
           Age  Cabin
Survived             
0         22.8   87.6
1         15.2   60.2 



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
61,62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,B28,NaN
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,B28,NaN


#### Registros atípicos

Se exploran filas completas de dos formas:

1. Cuántos atributos numéricos (`Age`, `Fare`, `SibSp`, `Parch`) tiene cada fila fuera de los límites de Tukey; las filas con 2 o más combinan valores extremos.
2. Tarifas atípicas **dentro de su propia clase** (Tukey calculado por `Pclass`), para encontrar casos como una tarifa altísima en tercera clase.

In [8]:
# 1) Número de atributos atípicos por fila
atipicos_por_attr = pd.DataFrame({col: mascara_tukey(df[col]) for col in NUM_TUKEY})
n_atip_fila = atipicos_por_attr.sum(axis=1)
print("Filas según número de atributos atípicos:")
print(n_atip_fila.value_counts().sort_index())

registros_multi = df.loc[n_atip_fila >= 2, ["PassengerId", "Pclass", "Age", "SibSp", "Parch", "Fare", "Survived"]]
print(f"\nFilas con 2 o más atributos atípicos: {len(registros_multi)}")
registros_multi.head(10)

Filas según número de atributos atípicos:
0    598
1    211
2     71
3     11
Name: count, dtype: int64

Filas con 2 o más atributos atípicos: 82


,PassengerId,Pclass,Age,SibSp,Parch,Fare,Survived
7,8,3,2.0,3,1,21.0750,0
16,17,3,2.0,4,1,29.1250,0
24,25,3,8.0,3,1,21.0750,0
27,28,1,19.0,3,2,263.0000,0
50,51,3,7.0,4,1,39.6875,0
54,55,1,65.0,0,1,61.9792,0
59,60,3,11.0,5,2,46.9000,0
63,64,3,4.0,3,2,27.9000,0
68,69,3,17.0,4,2,7.9250,1
71,72,3,16.0,5,2,46.9000,0


In [9]:
# 2) Tarifa atípica dentro de su propia clase
fare_atip_clase = df.groupby("Pclass")["Fare"].transform(mascara_tukey)
print("Tarifas atípicas dentro de su clase, por Pclass:")
print(df.loc[fare_atip_clase].groupby("Pclass").size())

# Las tarifas atípicas más altas de tercera clase
df.loc[fare_atip_clase & (df["Pclass"] == 3), ["PassengerId", "Pclass", "Fare", "Ticket", "SibSp", "Parch"]] \
    .sort_values("Fare", ascending=False).head(10)

Tarifas atípicas dentro de su clase, por Pclass:
Pclass
1    20
2     7
3    52
dtype: int64


,PassengerId,Pclass,Fare,Ticket,SibSp,Parch
201,202,3,69.5500,CA. 2343,8,2
180,181,3,69.5500,CA. 2343,8,2
159,160,3,69.5500,CA. 2343,8,2
324,325,3,69.5500,CA. 2343,8,2
792,793,3,69.5500,CA. 2343,8,2
863,864,3,69.5500,CA. 2343,8,2
846,847,3,69.5500,CA. 2343,8,2
692,693,3,56.4958,1601,0,0
169,170,3,56.4958,1601,0,0
643,644,3,56.4958,1601,0,0


#### Redundancia o nulo valor

Se calcula la correlación de Spearman entre las columnas numéricas (es robusta a atípicos y sirve para variables ordinales como `Pclass`). Además, se construye el tamaño de familia `FamilySize = SibSp + Parch + 1` para ver cuánto se solapa con `SibSp` y `Parch`.

In [10]:
# Correlación de Spearman (valor absoluto) entre columnas numéricas
corr_abs = df.select_dtypes("number").corr(method="spearman").abs()

# Para cada columna numérica, la otra columna con la que más se correlaciona
max_corr = {}
for col in corr_abs.columns:
    otra = corr_abs[col].drop(col).idxmax()
    max_corr[col] = f"{corr_abs.loc[col, otra]:.2f} ({otra})"

# SibSp y Parch frente al tamaño de familia
familia = df[["SibSp", "Parch"]].assign(FamilySize=df["SibSp"] + df["Parch"] + 1)
print("Spearman SibSp / Parch / FamilySize:")
print(familia.corr(method="spearman").round(2))

corr_abs.round(2)

Spearman SibSp / Parch / FamilySize:
            SibSp  Parch  FamilySize
SibSp        1.00   0.45        0.85
Parch        0.45   1.00        0.80
FamilySize   0.85   0.80        1.00


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
PassengerId,1.00,0.01,0.03,0.04,0.06,0.00,0.01
Survived,0.01,1.00,0.34,0.05,0.09,0.14,0.32
Pclass,0.03,0.34,1.00,0.36,0.04,0.02,0.69
Age,0.04,0.05,0.36,1.00,0.18,0.25,0.14
SibSp,0.06,0.09,0.04,0.18,1.00,0.45,0.45
Parch,0.00,0.14,0.02,0.25,0.45,1.00,0.41
Fare,0.01,0.32,0.69,0.14,0.45,0.41,1.00


#### Tabla resumen del diagnóstico

Se unen todas las verificaciones en una sola tabla. La columna **Registros atípicos** cuenta las filas en las que la columna es atípica y que además tienen 2 o más atributos atípicos (para `Fare` se añade cuántas tarifas son atípicas dentro de su clase).

**Borrador del agente, pendiente de revisión del grupo:** los textos de `MECANISMO` y `MITIGACION` en la siguiente celda los propuso el agente a partir de las salidas de arriba. El grupo debe revisarlos y editarlos antes de la entrega.

In [11]:
# BORRADOR DEL AGENTE: mecanismo probable de los faltantes (solo columnas con nulos)
MECANISMO = {
    "Age": "MAR: el % de nulos cambia según Pclass (27,7 % en 3.ª vs 6,0 % en 2.ª) y Survived",
    "Cabin": "MAR (posible MNAR): faltan 97,6 % en 3.ª clase vs 18,5 % en 1.ª",
    "Embarked": "MCAR: solo 2 filas, sin patrón visible (ambas de 1.ª clase, misma cabina B28)",
}

# BORRADOR DEL AGENTE: acción de mitigación propuesta para cada columna
MITIGACION = {
    "PassengerId": "Identificador: conservar como clave, excluir del análisis/modelado",
    "Survived": "Variable objetivo: sin cambios",
    "Pclass": "Sin cambios; se mantiene junto a Fare pese a rho = 0,69 (miden cosas distintas)",
    "Name": "Identificador: extraer el título (Mr, Mrs, Miss...) y no usar el nombre completo",
    "Sex": "Sin cambios; codificar como categórica",
    "Age": "Imputar con la mediana por grupo (Pclass y Sex) y agregar indicador de edad faltante; atípicos son edades plausibles: conservar",
    "SibSp": "Atípicos son familias grandes reales: conservar; resumir con FamilySize por redundancia",
    "Parch": "IQR = 0 marca como atípico todo valor distinto de 0: conservar; resumir con FamilySize",
    "Ticket": "Tratar 'LINE' como faltante; no usar como identificador (681 únicos en 891 filas)",
    "Fare": "Convertir los 15 ceros a NaN e imputar con la mediana por Pclass; atípicos reales: conservar o transformar con log",
    "Cabin": "Demasiados nulos para imputar: eliminar la columna y crear indicador TieneCabina (opcional: cubierta)",
    "Embarked": "Imputar con la moda (2 filas)",
}

# Registros atípicos: filas donde la columna es atípica y la fila tiene 2+ atributos atípicos
registros_atip = {}
for col in NUM_TUKEY:
    texto = f"{int((atipicos_por_attr[col] & (n_atip_fila >= 2)).sum())} filas con 2+ atípicos"
    if col == "Fare":
        texto += f"; {int(fare_atip_clase.sum())} atípicas dentro de su clase"
    registros_atip[col] = texto

tabla_diagnostico = diagnostico.assign(
    **{
        "Mecanismo": pd.Series(MECANISMO),
        "Registros atípicos": pd.Series(registros_atip),
        "Max |rho| (con)": pd.Series(max_corr),
        "Acción de mitigación": pd.Series(MITIGACION),
    }
)

# Orden de columnas según las verificaciones; la acción queda al final
tabla_diagnostico = tabla_diagnostico[[
    "dtype", "N únicos", "Único/constante",
    "Nulos", "% nulos", "Mecanismo",
    "Disfrazados", "Criterio disfrazado",
    "Tukey inf", "Tukey sup", "Atípicos", "% atípicos",
    "Registros atípicos", "Max |rho| (con)",
    "Acción de mitigación",
]].fillna("—")

# Mostrar los textos largos completos
with pd.option_context("display.max_colwidth", None):
    display(tabla_diagnostico)

,dtype,N únicos,Único/constante,Nulos,% nulos,Mecanismo,Disfrazados,Criterio disfrazado,Tukey inf,Tukey sup,Atípicos,% atípicos,Registros atípicos,Max |rho| (con),Acción de mitigación
Columna,,,,,,,,,,,,,,,
PassengerId,int64,891,Un valor por fila,0,0.00,—,0,no aplica,—,—,—,—,—,0.06 (SibSp),"Identificador: conservar como clave, excluir del análisis/modelado"
Survived,int64,2,—,0,0.00,—,0,no aplica,—,—,—,—,—,0.34 (Pclass),Variable objetivo: sin cambios
Pclass,int64,3,—,0,0.00,—,0,no aplica,—,—,—,—,—,0.69 (Fare),"Sin cambios; se mantiene junto a Fare pese a rho = 0,69 (miden cosas distintas)"
Name,object,891,Un valor por fila,0,0.00,—,0,marcador,—,—,—,—,—,—,"Identificador: extraer el título (Mr, Mrs, Miss...) y no usar el nombre completo"
Sex,object,2,—,0,0.00,—,0,marcador,—,—,—,—,—,—,Sin cambios; codificar como categórica
Age,float64,88,—,177,19.87,"MAR: el % de nulos cambia según Pclass (27,7 % en 3.ª vs 6,0 % en 2.ª) y Survived",0,valor 0,-6.69,64.81,11.0,1.23,2 filas con 2+ atípicos,0.36 (Pclass),Imputar con la mediana por grupo (Pclass y Sex) y agregar indicador de edad faltante; atípicos son edades plausibles: conservar
SibSp,int64,7,—,0,0.00,—,0,no aplica,-1.5,2.5,46.0,5.16,44 filas con 2+ atípicos,0.45 (Parch),Atípicos son familias grandes reales: conservar; resumir con FamilySize por redundancia
Parch,int64,7,—,0,0.00,—,0,no aplica,0.0,0.0,213.0,23.91,82 filas con 2+ atípicos,0.45 (SibSp),IQR = 0 marca como atípico todo valor distinto de 0: conservar; resumir con FamilySize
Ticket,object,681,—,0,0.00,—,4,marcador o ticket sin dígitos,—,—,—,—,—,—,Tratar 'LINE' como faltante; no usar como identificador (681 únicos en 891 filas)


### 4.3 Complemento del diagnóstico

Las celdas anteriores muestran varias cifras redondeadas. A partir de aquí todos los flotantes se muestran **con todos sus decimales**, y se recalculan las cifras clave del diagnóstico sobre el dataset original (`df`). También se verifica un hallazgo de los registros atípicos (que `Fare` es el precio del tiquete y no por persona) y se señala la **columna más problemática**, entregable del Ejercicio 1.

In [12]:
# Mostrar los flotantes con todos sus decimales (sin redondeo) en las tablas de pandas
pd.set_option("display.float_format", lambda x: str(float(x)))

#### 4.3.1 Valores faltantes y su mecanismo

In [13]:
# Nulos y % de nulos del dataset original (solo columnas con nulos)
nulos_original = df.isna().sum()
display(pd.DataFrame({
    "Nulos": nulos_original,
    "% nulos": 100 * nulos_original / len(df),
}).loc[nulos_original > 0])

# % de nulos de Age y Cabin según Pclass y según Survived (sin redondeo)
for grupo in ["Pclass", "Survived"]:
    print(f"% de nulos por {grupo}:")
    display(100 * df[["Age", "Cabin"]].isna().groupby(df[grupo]).mean())

,Nulos,% nulos
Age,177,19.865319865319865
Cabin,687,77.10437710437711
Embarked,2,0.2244668911335578


% de nulos por Pclass:


,Age,Cabin
Pclass,,
1,13.88888888888889,18.51851851851852
2,5.978260869565218,91.30434782608695
3,27.698574338085542,97.5560081466395


% de nulos por Survived:


,Age,Cabin
Survived,,
0,22.768670309653917,87.61384335154827
1,15.204678362573098,60.23391812865497


#### 4.3.2 Faltantes disfrazados y atípicos por atributo (Tukey)

In [14]:
# Faltantes disfrazados
print("Fare = 0:", int((df["Fare"] == 0).sum()), "filas |", 100 * (df["Fare"] == 0).sum() / len(df), "% de las filas")
print("Tiquetes sin dígitos:", int((~df["Ticket"].str.contains(r"\d")).sum()))
print("Valores distintos de Ticket:", df["Ticket"].nunique())

# Límites de Tukey y atípicos con todos sus decimales (funciones del Ejercicio 1)
tukey_completo = {}
for col in NUM_TUKEY:
    li, ls = limites_tukey(df[col])
    n_atip = int(mascara_tukey(df[col]).sum())
    tukey_completo[col] = {"Límite inferior": li, "Límite superior": ls, "Atípicos": n_atip, "% atípicos": 100 * n_atip / len(df)}
pd.DataFrame(tukey_completo).T.astype({"Atípicos": int})

Fare = 0: 15 filas | 1.6835016835016836 % de las filas
Tiquetes sin dígitos: 4
Valores distintos de Ticket: 681


,Límite inferior,Límite superior,Atípicos,% atípicos
Age,-6.6875,64.8125,11,1.2345679012345678
Fare,-26.724,65.6344,116,13.019079685746352
SibSp,-1.5,2.5,46,5.162738496071829
Parch,0.0,0.0,213,23.905723905723907


#### 4.3.3 Registros atípicos y tarifa por tiquete

En 4.2 aparecieron tarifas atípicas dentro de 3.ª clase repetidas en varios pasajeros del mismo tiquete (por ejemplo, `CA. 2343`). Se verifica si `Fare` es el precio del **tiquete completo**: si es así, todos los pasajeros de un mismo tiquete tienen la misma tarifa.

In [15]:
# Composición de las filas con 2 o más atributos atípicos (n_atip_fila viene de 4.2)
registros_2mas = df.loc[n_atip_fila >= 2]
print("Filas con 2 o más atributos atípicos:", len(registros_2mas))
print("Por clase:", registros_2mas["Pclass"].value_counts().sort_index().to_dict())
print("Con 3 o más hermanos/cónyuges (SibSp >= 3):", int((registros_2mas["SibSp"] >= 3).sum()))
print("Menores de 18 años:", int((registros_2mas["Age"] < 18).sum()))

Filas con 2 o más atributos atípicos: 82
Por clase: {1: 41, 3: 41}
Con 3 o más hermanos/cónyuges (SibSp >= 3): 44
Menores de 18 años: 37


In [16]:
# ¿Fare es por tiquete? Tarifas distintas dentro de cada tiquete compartido
pasajeros_por_tiquete = df["Ticket"].map(df["Ticket"].value_counts())
tarifas_distintas = df.groupby("Ticket")["Fare"].nunique()
print("Tiquetes compartidos por 2 o más pasajeros:", int((df["Ticket"].value_counts() > 1).sum()))
print("Pasajeros con tiquete compartido:", int((pasajeros_por_tiquete > 1).sum()))
print("Tiquetes con más de una tarifa distinta:", int((tarifas_distintas > 1).sum()))

# Tarifas atípicas dentro de 3.ª clase (fare_atip_clase viene de 4.2) que son de tiquetes compartidos
atip_tercera = fare_atip_clase & (df["Pclass"] == 3)
print("\nTarifas atípicas dentro de 3.ª clase:", int(atip_tercera.sum()))
print("De ellas, con tiquete compartido:", int((atip_tercera & (pasajeros_por_tiquete > 1)).sum()))

Tiquetes compartidos por 2 o más pasajeros: 134
Pasajeros con tiquete compartido: 344
Tiquetes con más de una tarifa distinta: 1

Tarifas atípicas dentro de 3.ª clase: 52
De ellas, con tiquete compartido: 52


#### 4.3.4 Redundancia en el dataset original

In [17]:
# Correlación de Spearman sin redondeo, sobre el dataset original
print("Spearman Pclass - Fare:", df["Pclass"].corr(df["Fare"], method="spearman"))
familia_original = df[["SibSp", "Parch"]].assign(FamilySize=df["SibSp"] + df["Parch"] + 1)
familia_original.corr(method="spearman")

Spearman Pclass - Fare: -0.6880316726256098


,SibSp,Parch,FamilySize
SibSp,1.0,0.45001397100861634,0.8491645107690946
Parch,0.45001397100861634,1.0,0.8012240815976783
FamilySize,0.8491645107690946,0.8012240815976783,1.0


#### 4.3.5 Evidencia sobre `Cabin`

In [18]:
# Cabin: valores observados, valores distintos y qué quedaría si se eliminaran las filas con nulo
print("Valores observados de Cabin:", int(df["Cabin"].notna().sum()), "de", len(df))
print("Valores distintos de Cabin:", df["Cabin"].nunique())

filas_con_cabina = df.loc[df["Cabin"].notna()]
print("\nSi se eliminaran las filas sin Cabin, quedarían por clase:")
display(pd.DataFrame({
    "Pasajeros": filas_con_cabina["Pclass"].value_counts().sort_index(),
    "% del subconjunto": 100 * filas_con_cabina["Pclass"].value_counts(normalize=True).sort_index(),
}))

# Supervivencia según tenga o no cabina registrada
supervivencia_cabina = 100 * df["Survived"].groupby(df["Cabin"].notna()).mean()
supervivencia_cabina.index = ["Sin cabina", "Con cabina"]
supervivencia_cabina.rename("% sobrevivió").to_frame()

Valores observados de Cabin: 204 de 891
Valores distintos de Cabin: 147

Si se eliminaran las filas sin Cabin, quedarían por clase:


,Pasajeros,% del subconjunto
Pclass,,
1,176,86.27450980392157
2,16,7.8431372549019605
3,12,5.88235294117647


,% sobrevivió
Sin cabina,29.985443959243085
Con cabina,66.66666666666666


### 4.4 Columna más problemática: `Cabin`

`Cabin` es la columna más problemática del dataset, por tres razones que se ven en las salidas de 4.3.1 y 4.3.5:

1. **Volumen de faltantes:** le falta el 77.10437710437711 % de los valores (687 de 891). Es, por mucho, la columna con más nulos.
2. **Faltantes no aleatorios (MAR, posible MNAR):** falta en el 97.5560081466395 % de los pasajeros de 3.ª clase frente al 18.51851851851852 % de 1.ª, y en el 87.61384335154827 % de quienes no sobrevivieron frente al 60.23391812865497 % de quienes sí. Cualquier tratamiento ingenuo sesga los datos.
3. **Ninguna corrección es buena:**
   - Imputarla obligaría a inventar más de tres cuartas partes de una columna con 147 valores distintos.
   - Eliminar las filas con nulo dejaría solo 204 pasajeros, el 86.27450980392157 % de ellos de 1.ª clase.
   - Eliminar la columna pierde la información. Lo único recuperable es si el pasajero tenía cabina registrada, y ese indicador trae riesgo de *data leakage*: sobrevivió el 66.66666666666666 % de quienes tenían cabina frente al 29.985443959243085 % de quienes no.

Las demás columnas con problemas tienen tratamientos razonables: a `Age` le falta el 19.865319865319865 % y se puede imputar por grupo; los atípicos de `Parch` (23.905723905723907 %) son un efecto de su IQR igual a 0; y los atípicos de `Fare` no son errores. 4.3.3 muestra que `Fare` es el precio del tiquete completo: de 134 tiquetes compartidos, solo 1 tiene más de una tarifa, y las 52 tarifas atípicas dentro de 3.ª clase son todas de tiquetes compartidos.

## Ejercicio 2: Limpieza del dataset

La limpieza aplica, en orden, las acciones de mitigación que propuso el diagnóstico (columna **Acción de mitigación** de `tabla_diagnostico` y hallazgos de 4.3). Cada sección empieza por el hallazgo que atiende. Se trabaja sobre una copia, `df_limpio`; el original `df` no se modifica, para comparar siempre antes y después.

**Trazabilidad: hallazgo del Ejercicio 1 → acción → sección**

| Hallazgo del diagnóstico | Acción en la limpieza | Sección |
|---|---|---|
| `PassengerId` tiene un valor por fila: un pasajero repetido no se vería como duplicado | Buscar duplicados ignorando `PassengerId` (el diagnóstico no revisó duplicados) | 5.1 |
| Faltantes disfrazados: 15 `Fare = 0` y 4 tiquetes `"LINE"` | Convertirlos a `NaN` con Boolean Indexing | 5.2 |
| `Cabin`: 77 % de nulos, MAR (posible MNAR); columna más problemática | Crear `TieneCabina` y eliminar la columna con `dropna` | 5.3 |
| `Age`: 20 % de nulos, MAR (depende de `Pclass` y `Survived`) | Imputar con la mediana por `Pclass` × `Sex` y crear `EdadFaltante` | 5.3 |
| `Embarked`: 2 nulos, MCAR | Imputar con la moda | 5.3 |
| `Fare`: ceros convertidos a `NaN`, tarifa ligada a la clase | Imputar con la mediana por `Pclass` | 5.3 |
| `Fare`: 116 atípicos de Tukey, reales (propuesta: conservar o transformar) | Conservar las filas y acotar con winsorización p5–p95 | 5.4 |
| Atípicos de `Age`, `SibSp`, `Parch` plausibles; IQR de `Parch` = 0 | Conservarlos | 5.4 |
| Registros atípicos: 82 filas con 2+ atributos atípicos; `Fare` es precio por tiquete | Conservar las filas; evaluar la tarifa por persona y guardar `pasajeros_tiquete` | 5.4 |
| `SibSp` y `Parch` se solapan con el tamaño de familia | Crear `tamano_familia` y eliminar `SibSp` y `Parch` | 5.5 |
| `Name` identificador; `Ticket` casi identificador | Extraer `Titulo` y eliminar `Name` y `Ticket` | 5.5 |
| `PassengerId` identificador; `Pclass`–`Fare` correlacionadas pero distintas | Conservar `PassengerId` como clave; conservar `Pclass` y `Fare` | 5.5 |
| `Sex`: codificar como categórica | Convertir `Sex`, `Embarked` y `Titulo` a tipo `category` | 5.5 |

In [19]:
# Copia de trabajo: df (original) queda intacto para comparar antes y después
df_limpio = df.copy()

# Registro del shape después de cada paso, para el resumen final
registro_pasos = [("Original", df_limpio.shape)]
print("shape inicial:", df_limpio.shape)

shape inicial: (891, 12)


### 5.1 Duplicados (`df.drop_duplicates`)

**Hallazgo:** el diagnóstico no revisó duplicados, y mostró que `PassengerId` tiene un valor distinto en cada fila. Por eso un mismo pasajero registrado dos veces no aparecería como duplicado exacto.

**Acción:** buscar duplicados exactos y también ignorando `PassengerId`; eliminar con el criterio más estricto (el segundo), conservando la primera aparición, y reportar el shape antes y después.

In [20]:
# 1) Duplicados exactos (todas las columnas)
n_dup_exactos = int(df_limpio.duplicated().sum())

# 2) Duplicados ignorando el identificador
cols_sin_id = df_limpio.columns.drop("PassengerId")
n_dup_sin_id = int(df_limpio.duplicated(subset=cols_sin_id).sum())

print("Duplicados exactos:", n_dup_exactos)
print("Duplicados ignorando PassengerId:", n_dup_sin_id)

# Eliminación con el criterio más estricto
shape_antes = df_limpio.shape
df_limpio = df_limpio.drop_duplicates(subset=cols_sin_id, keep="first")
print("\nshape antes de drop_duplicates:  ", shape_antes)
print("shape después de drop_duplicates:", df_limpio.shape)
print("Filas eliminadas:", shape_antes[0] - df_limpio.shape[0])

registro_pasos.append(("5.1 drop_duplicates", df_limpio.shape))

Duplicados exactos: 0
Duplicados ignorando PassengerId: 0

shape antes de drop_duplicates:   (891, 12)
shape después de drop_duplicates: (891, 12)
Filas eliminadas: 0


### 5.2 Inconsistencias (Boolean Indexing)

**Hallazgo:** la verificación de faltantes disfrazados encontró 15 tarifas en 0 (una tarifa de 0 no es plausible) y 4 tiquetes sin ningún dígito (`"LINE"`), y la tabla de mitigación propuso tratarlos como faltantes.

**Acción:** filtrar esos subconjuntos con máscaras booleanas y convertirlos a `NaN`. `Fare` se imputa en 5.3; `Ticket` se decide en 5.5. Además se verifican reglas de dominio que el diagnóstico no cubrió.

In [21]:
# Subconjunto con tarifa 0 (faltante disfrazado)
mask_fare_cero = df_limpio["Fare"] == 0
print("Filas con Fare = 0:", int(mask_fare_cero.sum()))
df_limpio.loc[mask_fare_cero, ["PassengerId", "Pclass", "Ticket", "Fare", "Age", "Survived"]]

Filas con Fare = 0: 15


,PassengerId,Pclass,Ticket,Fare,Age,Survived
179,180,3,LINE,0.0,36.0,0
263,264,1,112059,0.0,40.0,0
271,272,3,LINE,0.0,25.0,1
277,278,2,239853,0.0,NaN,0
302,303,3,LINE,0.0,19.0,0
413,414,2,239853,0.0,NaN,0
466,467,2,239853,0.0,NaN,0
481,482,2,239854,0.0,NaN,0
597,598,3,LINE,0.0,49.0,0
633,634,1,112052,0.0,NaN,0


In [22]:
# Subconjunto de tiquetes sin ningún dígito
mask_ticket_sin_digitos = ~df_limpio["Ticket"].str.contains(r"\d")
print("Tiquetes sin dígitos:", int(mask_ticket_sin_digitos.sum()))
print(df_limpio.loc[mask_ticket_sin_digitos, "Ticket"].value_counts())

# Corrección: ambos faltantes disfrazados pasan a NaN
df_limpio.loc[mask_fare_cero, "Fare"] = np.nan
df_limpio.loc[mask_ticket_sin_digitos, "Ticket"] = np.nan

print("\nNulos en Fare tras la corrección:  ", int(df_limpio["Fare"].isna().sum()))
print("Nulos en Ticket tras la corrección:", int(df_limpio["Ticket"].isna().sum()))

Tiquetes sin dígitos: 4
Ticket
LINE    4
Name: count, dtype: int64

Nulos en Fare tras la corrección:   15
Nulos en Ticket tras la corrección: 4


In [23]:
# Reglas de dominio: cada máscara marca las filas que incumplen la regla
reglas = {
    "Sex fuera de {male, female}": ~df_limpio["Sex"].isin(["male", "female"]),
    "Pclass fuera de {1, 2, 3}": ~df_limpio["Pclass"].isin([1, 2, 3]),
    "Survived fuera de {0, 1}": ~df_limpio["Survived"].isin([0, 1]),
    "Embarked fuera de {S, C, Q} (sin contar nulos)": df_limpio["Embarked"].notna() & ~df_limpio["Embarked"].isin(["S", "C", "Q"]),
    "Age <= 0 o Age > 100": (df_limpio["Age"] <= 0) | (df_limpio["Age"] > 100),
    "SibSp o Parch negativos": (df_limpio["SibSp"] < 0) | (df_limpio["Parch"] < 0),
    "Fare negativa": df_limpio["Fare"] < 0,
}
registro_pasos.append(("5.2 Boolean Indexing", df_limpio.shape))
pd.Series({regla: int(mascara.sum()) for regla, mascara in reglas.items()}, name="Filas que incumplen")

Sex fuera de {male, female}                       0
Pclass fuera de {1, 2, 3}                         0
Survived fuera de {0, 1}                          0
Embarked fuera de {S, C, Q} (sin contar nulos)    0
Age <= 0 o Age > 100                              0
SibSp o Parch negativos                           0
Fare negativa                                     0
Name: Filas que incumplen, dtype: int64

### 5.3 Faltantes (punto 8)

**Hallazgo:** `Cabin` (MAR, posible MNAR), `Age` (MAR) y `Embarked` (MCAR) tienen nulos; tras 5.2, también `Fare`. Se cuentan los nulos actuales y se revisa en qué clases quedaron los ceros de `Fare`.

In [24]:
# Nulos por columna después de 5.2 (solo las columnas con nulos)
nulos = df_limpio.isna().sum()
tabla_nulos = pd.DataFrame({"Nulos": nulos, "% nulos": 100 * nulos / len(df_limpio)})
display(tabla_nulos.loc[tabla_nulos["Nulos"] > 0])

# Distribución por clase de los Fare = 0 convertidos a NaN
pd.DataFrame({
    "Fare nulos": df_limpio["Fare"].isna().groupby(df_limpio["Pclass"]).sum(),
    "% de la clase": 100 * df_limpio["Fare"].isna().groupby(df_limpio["Pclass"]).mean(),
})

,Nulos,% nulos
Age,177,19.865319865319865
Ticket,4,0.4489337822671156
Fare,15,1.6835016835016836
Cabin,687,77.10437710437711
Embarked,2,0.2244668911335578


,Fare nulos,% de la clase
Pclass,,
1,5,2.314814814814815
2,6,3.260869565217391
3,4,0.8146639511201629


**Decisión por columna**, según el mecanismo del diagnóstico:

| Columna | Mecanismo | Decisión | Justificación |
|---|---|---|---|
| `Cabin` | MAR (posible MNAR) | **Eliminar la columna** con `df.dropna(axis=1, thresh=...)`, guardando antes el indicador `TieneCabina` | Es la columna más problemática (4.4): imputarla inventaría la mayoría de sus valores y borrar sus filas dejaría casi solo pasajeros de 1.ª clase. Como el faltante depende de la clase, *tener o no cabina* sí es información y se conserva como indicador. No se extrae la cubierta (propuesta opcional del diagnóstico) porque faltaría en las mismas filas. |
| `Age` | MAR (depende de `Pclass` y `Survived`) | **Imputar** con la mediana por grupo (`Pclass` × `Sex`) con `df.fillna`, y crear el indicador `EdadFaltante` | Eliminar las filas perdería casi una de cada cinco y, al ser MAR, sesgaría la muestra hacia las clases con menos faltantes. Bajo MAR es válido imputar condicionando en variables observadas; la mediana es robusta a los atípicos. El indicador conserva la señal de que el dato faltaba. |
| `Embarked` | MCAR | **Imputar** con la moda con `df.fillna` | Son 2 filas sin patrón: una imputación simple no sesga el conjunto y no se pierden filas. Como ambas comparten tiquete y cabina (4.2), quedan con el mismo puerto. |
| `Fare` (ceros → `NaN`) | MAR | **Imputar** con la mediana por `Pclass` con `df.fillna` | Los ceros no se reparten igual entre clases (tabla de arriba) y la tarifa está muy ligada a la clase (4.3.4), así que se imputa condicionando en `Pclass`. |
| `Ticket` (`"LINE"` → `NaN`) | — | Sin imputar | La columna se decide en 5.5. |

In [25]:
# 5.3.1 Cabin: indicador de cabina registrada y eliminación de la columna con dropna
df_limpio["TieneCabina"] = df_limpio["Cabin"].notna().astype(int)

# Umbral: se conservan solo las columnas con al menos 50 % de valores no nulos
UMBRAL_NO_NULOS = 0.5
min_no_nulos = int(np.ceil(UMBRAL_NO_NULOS * len(df_limpio)))
print("Mínimo de valores no nulos para conservar una columna:", min_no_nulos)
print("Valores no nulos por columna:")
print(df_limpio.notna().sum())

shape_antes = df_limpio.shape
cols_antes = df_limpio.columns
df_limpio = df_limpio.dropna(axis=1, thresh=min_no_nulos)

print("\nColumnas eliminadas por dropna:", list(cols_antes.difference(df_limpio.columns)))
print("shape antes de dropna:  ", shape_antes)
print("shape después de dropna:", df_limpio.shape)
registro_pasos.append(("5.3.1 dropna (Cabin) + TieneCabina", df_limpio.shape))

Mínimo de valores no nulos para conservar una columna: 446
Valores no nulos por columna:
PassengerId    891
Survived       891
Pclass         891
Name           891
Sex            891
Age            714
SibSp          891
Parch          891
Ticket         887
Fare           876
Cabin          204
Embarked       889
TieneCabina    891
dtype: int64

Columnas eliminadas por dropna: ['Cabin']
shape antes de dropna:   (891, 13)
shape después de dropna: (891, 12)


In [26]:
# 5.3.2 Embarked: imputación con la moda
moda_embarked = df_limpio["Embarked"].mode()[0]
print("Frecuencia de cada puerto antes de imputar:")
print(df_limpio["Embarked"].value_counts())
print("\nModa de Embarked:", moda_embarked)

filas_embarked_nulo = df_limpio.index[df_limpio["Embarked"].isna()]
df_limpio["Embarked"] = df_limpio["Embarked"].fillna(moda_embarked)

# Filas imputadas
df_limpio.loc[filas_embarked_nulo, ["PassengerId", "Pclass", "Ticket", "Fare", "Embarked"]]

Frecuencia de cada puerto antes de imputar:
Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

Moda de Embarked: S


,PassengerId,Pclass,Ticket,Fare,Embarked
61,62,1,113572,80.0,S
829,830,1,113572,80.0,S


In [27]:
# 5.3.3 Fare: imputación de los ceros (ahora NaN) con la mediana de su clase
mediana_fare_clase = df_limpio.groupby("Pclass")["Fare"].median()
print("Mediana de Fare por Pclass (sin los ceros):")
print(mediana_fare_clase)

filas_fare_nulo = df_limpio.index[df_limpio["Fare"].isna()]
df_limpio["Fare"] = df_limpio["Fare"].fillna(df_limpio["Pclass"].map(mediana_fare_clase))

# Filas imputadas: Fare original (0) y Fare imputada
pd.DataFrame({
    "Pclass": df_limpio.loc[filas_fare_nulo, "Pclass"],
    "Fare original": df.loc[filas_fare_nulo, "Fare"],
    "Fare imputada": df_limpio.loc[filas_fare_nulo, "Fare"],
})

Mediana de Fare por Pclass (sin los ceros):
Pclass
1   61.9792
2   15.0229
3      8.05
Name: Fare, dtype: float64


,Pclass,Fare original,Fare imputada
179,3,0.0,8.05
263,1,0.0,61.9792
271,3,0.0,8.05
277,2,0.0,15.0229
302,3,0.0,8.05
413,2,0.0,15.0229
466,2,0.0,15.0229
481,2,0.0,15.0229
597,3,0.0,8.05
633,1,0.0,61.9792


In [28]:
# 5.3.4 Age: indicador de faltante e imputación con la mediana por Pclass y Sex
df_limpio["EdadFaltante"] = df_limpio["Age"].isna().astype(int)

mediana_age_grupo = df_limpio.groupby(["Pclass", "Sex"])["Age"].median()
print("Mediana de Age por Pclass y Sex:")
print(mediana_age_grupo)

# Nulos a imputar en cada grupo
print("\nNulos de Age por grupo:")
print(df_limpio["Age"].isna().groupby([df_limpio["Pclass"], df_limpio["Sex"]]).sum())

age_antes = df_limpio["Age"].copy()
df_limpio["Age"] = df_limpio["Age"].fillna(
    df_limpio.groupby(["Pclass", "Sex"])["Age"].transform("median")
)
print("\nNulos de Age después de imputar:", int(df_limpio["Age"].isna().sum()))

Mediana de Age por Pclass y Sex:
Pclass  Sex   
1       female   35.0
        male     40.0
2       female   28.0
        male     30.0
3       female   21.5
        male     25.0
Name: Age, dtype: float64

Nulos de Age por grupo:
Pclass  Sex   
1       female     9
        male      21
2       female     2
        male       9
3       female    42
        male      94
Name: Age, dtype: int64

Nulos de Age después de imputar: 0


**Comparación de `Age` antes y después de imputar.** La media y la mediana *antes* se calculan sobre los valores observados (sin nulos); *después*, sobre la columna completa. También se incluye la desviación estándar, porque imputar con medianas concentra valores en pocos puntos y tiende a reducir la dispersión.

In [29]:
# Media, mediana y desviación estándar de Age antes y después de imputar
comparacion_age = pd.DataFrame({
    "Antes de imputar": [age_antes.mean(), age_antes.median(), age_antes.std()],
    "Después de imputar": [df_limpio["Age"].mean(), df_limpio["Age"].median(), df_limpio["Age"].std()],
}, index=["Media", "Mediana", "Desviación estándar"])
comparacion_age["Diferencia (después - antes)"] = comparacion_age["Después de imputar"] - comparacion_age["Antes de imputar"]

print("Valores no nulos antes:", int(age_antes.notna().sum()), "| después:", int(df_limpio["Age"].notna().sum()))
comparacion_age

Valores no nulos antes: 714 | después: 891


,Antes de imputar,Después de imputar,Diferencia (después - antes)
Media,29.69911764705882,29.11242424242424,-0.5866934046345804
Mediana,28.0,26.0,-2.0
Desviación estándar,14.526497332334044,13.304424074688153,-1.2220732576458904


In [30]:
# Figura: distribución de Age antes y después de imputar (mismos bins para comparar)
RUTA_FIGURAS = Path("figuras")
RUTA_FIGURAS.mkdir(exist_ok=True)

bins_age = np.arange(0, 85, 5)
fig, ejes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
ejes[0].hist(age_antes.dropna(), bins=bins_age, color="#8C8C8C", edgecolor="white")
ejes[0].set_title("Age antes de imputar (solo observados)")
ejes[1].hist(df_limpio["Age"], bins=bins_age, color="#2A6FDB", edgecolor="white")
ejes[1].set_title("Age después de imputar (mediana por Pclass y Sex)")
for eje in ejes:
    eje.set_xlabel("Edad (años)")
    eje.grid(axis="y", alpha=0.3)
ejes[0].set_ylabel("Número de pasajeros")
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / "age_antes_despues.png", dpi=120)
plt.show()

In [31]:
# Verificación: nulos restantes después del punto 8
print("Nulos por columna después del punto 8:")
print(df_limpio.isna().sum())
registro_pasos.append(("5.3 fillna (Embarked, Fare, Age) + EdadFaltante", df_limpio.shape))
print("\nshape tras 5.3:", df_limpio.shape)

Nulos por columna después del punto 8:
PassengerId     0
Survived        0
Pclass          0
Name            0
Sex             0
Age             0
SibSp           0
Parch           0
Ticket          4
Fare            0
Embarked        0
TieneCabina     0
EdadFaltante    0
dtype: int64

shape tras 5.3: (891, 13)


Los únicos nulos que quedan son los de `Ticket` (los tiquetes `"LINE"` de 5.2), que no se imputan porque la columna se decide en 5.5.

### 5.4 Atípicos (punto 9)

**Hallazgo:** `Fare` tiene 116 atípicos según Tukey (4.3.2), pero son tarifas reales (primera clase y tiquetes de grupo), y el diagnóstico propuso **conservarlos o transformarlos**, no eliminarlos. Los atípicos de `Age`, `SibSp` y `Parch` son valores plausibles (en `Parch`, un efecto de su IQR igual a 0).

**Acción:** el enunciado pide tratar `Fare` con winsorización (p5–p95) o con los límites de Tukey. Se elige la **winsorización** porque es la transformación más fiel a la propuesta del diagnóstico: no elimina filas, conserva el orden de las tarifas y solo acota las colas. Los límites de Tukey recortarían muchos más valores, porque su límite superior es bajo (la celda siguiente compara cuántos valores modifica cada criterio). Los atípicos de `Age`, `SibSp` y `Parch` se conservan sin cambios.

#### 5.4.1 Winsorización de `Fare`

Los percentiles se calculan sobre `Fare` ya corregida en 5.3 (ceros imputados).

In [32]:
# Fare después del punto 8 (ceros imputados), antes de winsorizar
fare_antes_winsor = df_limpio["Fare"].copy()

# Percentiles 5 y 95
p05, p95 = fare_antes_winsor.quantile([0.05, 0.95])
print("Percentil 5 de Fare: ", p05)
print("Percentil 95 de Fare:", p95)
print("Valores por debajo del p5 (se suben):", int((fare_antes_winsor < p05).sum()))
print("Valores por encima del p95 (se bajan):", int((fare_antes_winsor > p95).sum()))

# Comparación con Tukey (función limites_tukey del Ejercicio 1) sobre la misma serie
tukey_inf, tukey_sup = limites_tukey(fare_antes_winsor)
print("\nLímites de Tukey de Fare:", tukey_inf, "|", tukey_sup)
print("Valores que Tukey recortaría por encima:", int((fare_antes_winsor > tukey_sup).sum()))

# Winsorización
df_limpio["Fare"] = fare_antes_winsor.clip(lower=p05, upper=p95)

# La winsorización no altera el orden de las tarifas: la correlación de Spearman con la serie previa
print("\nSpearman entre Fare antes y después de winsorizar:", fare_antes_winsor.corr(df_limpio["Fare"], method="spearman"))

Percentil 5 de Fare:  7.2292
Percentil 95 de Fare: 112.07915
Valores por debajo del p5 (se suben): 40
Valores por encima del p95 (se bajan): 45

Límites de Tukey de Fare: -27.099999999999998 | 66.3
Valores que Tukey recortaría por encima: 116

Spearman entre Fare antes y después de winsorizar: 0.9998221135170301


#### 5.4.2 Media original frente a la tratada

Se compara la **media original** (`df["Fare"]`, sin ningún tratamiento) con la media después de imputar los ceros y con la media **tratada** (winsorizada).

In [33]:
# Estadísticos de Fare: original, tras imputar ceros y winsorizada
def resumen_fare(serie):
    return pd.Series({
        "Media": serie.mean(),
        "Mediana": serie.median(),
        "Desviación estándar": serie.std(),
        "Mínimo": serie.min(),
        "Máximo": serie.max(),
    })

comparacion_fare = pd.DataFrame({
    "Original (df)": resumen_fare(df["Fare"]),
    "Tras imputar ceros (5.3)": resumen_fare(fare_antes_winsor),
    "Winsorizada p5-p95": resumen_fare(df_limpio["Fare"]),
})
comparacion_fare["Diferencia (winsorizada - original)"] = comparacion_fare["Winsorizada p5-p95"] - comparacion_fare["Original (df)"]
comparacion_fare

,Original (df),Tras imputar ceros (5.3),Winsorizada p5-p95,Diferencia (winsorizada - original)
Media,32.204207968574636,32.68931840628507,28.22123552188552,-3.9829724466891143
Mediana,14.4542,14.5,14.5,0.04579999999999984
Desviación estándar,49.693428597180905,49.61163877217443,29.14891277694091,-20.544515820239994
Mínimo,0.0,4.0125,7.2292,7.2292
Máximo,512.3292,512.3292,112.07915,-400.25005


In [34]:
# Figura: distribución de Fare antes (original) y después (winsorizada)
fig, ejes = plt.subplots(2, 2, figsize=(12, 7), gridspec_kw={"height_ratios": [3, 1]})
series_fare = [
    ("Fare original (df)", df["Fare"], "#8C8C8C"),
    ("Fare winsorizada (p5-p95)", df_limpio["Fare"], "#2A6FDB"),
]
for col, (titulo, serie, color) in enumerate(series_fare):
    ejes[0, col].hist(serie, bins=40, color=color, edgecolor="white")
    ejes[0, col].set_title(titulo)
    ejes[0, col].set_ylabel("Número de pasajeros")
    ejes[0, col].grid(axis="y", alpha=0.3)
    ejes[1, col].boxplot(serie, vert=False, widths=0.6,
                         boxprops={"color": color}, medianprops={"color": color},
                         flierprops={"markeredgecolor": color})
    ejes[1, col].set_yticks([])
    ejes[1, col].set_xlabel("Tarifa (libras)")
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / "fare_antes_despues.png", dpi=120)
plt.show()

#### 5.4.3 Registros atípicos y tarifa por tiquete

**Hallazgo:** el diagnóstico encontró 82 filas con 2 o más atributos atípicos y tarifas atípicas dentro de 3.ª clase, y 4.3.3 mostró que `Fare` es el precio del tiquete completo (los pasajeros de un mismo tiquete pagan lo mismo) y que esas tarifas atípicas de 3.ª clase son de tiquetes compartidos.

**Acción:**

1. Evaluar si una **tarifa por persona** (`Fare` / pasajeros del tiquete) corrige los atípicos dentro de cada clase.
2. Conservar el tamaño del grupo como variable, `pasajeros_tiquete`, antes de eliminar `Ticket` en 5.5. Los 4 tiquetes `"LINE"` son faltantes (5.2), no un tiquete real compartido, así que se cuentan como 1 pasajero.
3. Verificar qué pasó con los registros atípicos del diagnóstico.

In [35]:
# Pasajeros del dataset que comparten cada tiquete ("LINE" ya es NaN: cuenta como 1)
df_limpio["pasajeros_tiquete"] = df_limpio["Ticket"].map(df_limpio["Ticket"].value_counts()).fillna(1).astype(int)
print("Distribución de pasajeros_tiquete:")
print(df_limpio["pasajeros_tiquete"].value_counts().sort_index())

# Tarifa por persona (sobre Fare antes de winsorizar) y atípicos de Tukey dentro de cada clase
tarifa_por_persona = fare_antes_winsor / df_limpio["pasajeros_tiquete"]
pd.DataFrame({
    "Atípicos de Fare por tiquete": fare_antes_winsor.groupby(df_limpio["Pclass"]).transform(mascara_tukey).groupby(df_limpio["Pclass"]).sum(),
    "Atípicos de tarifa por persona": tarifa_por_persona.groupby(df_limpio["Pclass"]).transform(mascara_tukey).groupby(df_limpio["Pclass"]).sum(),
}).rename_axis("Pclass (Tukey dentro de la clase)")

Distribución de pasajeros_tiquete:
pasajeros_tiquete
1    551
2    188
3     63
4     40
5     10
6     18
7     21
Name: count, dtype: int64


,Atípicos de Fare por tiquete,Atípicos de tarifa por persona
Pclass (Tukey dentro de la clase),,
1,20,9
2,7,28
3,52,103


La tarifa por persona **no** corrige los atípicos dentro de cada clase: los reduce en 1.ª clase, pero los aumenta en 2.ª y 3.ª (tabla de arriba). Una explicación posible es que el dataset no contiene a todos los pasajeros de cada tiquete, así que dividir entre los que aparecen distorsiona la tarifa. Por eso se descarta como corrección: `Fare` se conserva como precio del tiquete (winsorizado) y `pasajeros_tiquete` se guarda para que el modelo pueda interpretar esa tarifa según el tamaño del grupo.

In [36]:
# Registros atípicos del diagnóstico: siguen en el dataset y qué les hizo la winsorización
fare_modificada = fare_antes_winsor != df_limpio["Fare"]
multi_atipicos = n_atip_fila >= 2  # 4.2: filas con 2 o más atributos atípicos

print("Filas con 2+ atributos atípicos en el diagnóstico:", int(multi_atipicos.sum()))
print("Siguen en df_limpio:", int(df.loc[multi_atipicos, "PassengerId"].isin(df_limpio["PassengerId"]).sum()))
print("De ellas, con Fare acotada por la winsorización:", int((multi_atipicos & fare_modificada).sum()))

print("\nTarifas atípicas dentro de 3.ª clase (4.2):", int(atip_tercera.sum()))
print("De ellas, acotadas por la winsorización:", int((atip_tercera & fare_modificada).sum()))

# Atípicos de Tukey de Age, SibSp y Parch: se conservan sin cambios
print("\nAtípicos conservados sin cambios:")
for col in ["Age", "SibSp", "Parch"]:
    print(f"  {col}:", int(mascara_tukey(df[col]).sum()))

Filas con 2+ atributos atípicos en el diagnóstico: 82
Siguen en df_limpio: 82
De ellas, con Fare acotada por la winsorización: 25

Tarifas atípicas dentro de 3.ª clase (4.2): 52
De ellas, acotadas por la winsorización: 0

Atípicos conservados sin cambios:
  Age: 11
  SibSp: 46
  Parch: 213


In [37]:
registro_pasos.append(("5.4 winsorización Fare + pasajeros_tiquete", df_limpio.shape))
print("shape tras 5.4:", df_limpio.shape)

shape tras 5.4: (891, 14)


### 5.5 Redundancia y data leakage (punto 10)

**Hallazgo:** `SibSp` y `Parch` se solapan con el tamaño de familia (4.3.4); `Pclass` y `Fare` están correlacionadas pero miden cosas distintas; `Name` y `Ticket` son identificadores o casi identificadores; `Sex` debe codificarse como categórica.

#### 5.5.1 Variable derivada `tamano_familia` y correlaciones

Se crea `tamano_familia = SibSp + Parch + 1` y se calcula la correlación entre las variables numéricas, con Pearson (relación lineal) y con Spearman (por rangos, robusta a atípicos; la misma del diagnóstico).

In [38]:
# Variable derivada: tamaño de la familia a bordo (incluye al pasajero)
df_limpio["tamano_familia"] = df_limpio["SibSp"] + df_limpio["Parch"] + 1

# Comprobación: es exactamente la suma (diferencia máxima = 0)
print("Diferencia máxima con SibSp + Parch + 1:",
      int((df_limpio["tamano_familia"] - df_limpio["SibSp"] - df_limpio["Parch"] - 1).abs().max()))
print("\nDistribución de tamano_familia:")
print(df_limpio["tamano_familia"].value_counts().sort_index())

Diferencia máxima con SibSp + Parch + 1: 0

Distribución de tamano_familia:
tamano_familia
1     537
2     161
3     102
4      29
5      15
6      22
7      12
8       6
11      7
Name: count, dtype: int64


In [39]:
# Correlación de Pearson y de Spearman entre las numéricas (sin PassengerId, que es un identificador)
numericas = df_limpio.select_dtypes("number").drop(columns="PassengerId")
corr_pearson = numericas.corr(method="pearson")
print("Correlación de Pearson:")
display(corr_pearson)

corr_spearman = numericas.corr(method="spearman")
print("Correlación de Spearman:")
display(corr_spearman)

Correlación de Pearson:


,Survived,Pclass,Age,SibSp,Parch,Fare,TieneCabina,EdadFaltante,pasajeros_tiquete,tamano_familia
Survived,1.0,-0.33848103596101514,-0.059579368461496376,-0.035322498885735576,0.08162940708348335,0.30260369608265736,0.3169115231122961,-0.09219652324217223,0.041187898312162075,0.016638989282745233
Pclass,-0.33848103596101514,1.0,-0.41358298691271,0.08308136284568686,0.018442671310748508,-0.7144730047779287,-0.725541241828957,0.1729328590612901,-0.010881162151873253,0.06599690843099823
Age,-0.059579368461496376,-0.41358298691271,1.0,-0.24985377786683594,-0.17552611042957453,0.17601839958555596,0.2770218353669802,-0.08861789873661559,-0.22461361691810086,-0.25845640944373865
SibSp,-0.035322498885735576,0.08308136284568686,-0.24985377786683594,1.0,0.41483769862015624,0.24786257892446611,-0.04046001036349774,0.018957570883804405,0.6698336078352974,0.890711671869885
Parch,0.08162940708348335,0.018442671310748508,-0.17552611042957453,0.41483769862015624,1.0,0.25429138128275275,0.036987034298118136,-0.12410382550914598,0.6009172776437808,0.7831107750000581
Fare,0.30260369608265736,-0.7144730047779287,0.17601839958555596,0.24786257892446611,0.25429138128275275,1.0,0.6197414539613038,-0.11000793997917713,0.45644960234449294,0.29644532097653414
TieneCabina,0.3169115231122961,-0.725541241828957,0.2770218353669802,-0.04046001036349774,0.036987034298118136,0.6197414539613038,1.0,-0.1441108792288473,0.03359663134997703,-0.009174914777952252
EdadFaltante,-0.09219652324217223,0.1729328590612901,-0.08861789873661559,0.018957570883804405,-0.12410382550914598,-0.11000793997917713,-0.1441108792288473,1.0,-0.039641997689729334,-0.04904337230176844
pasajeros_tiquete,0.041187898312162075,-0.010881162151873253,-0.22461361691810086,0.6698336078352974,0.6009172776437808,0.45644960234449294,0.03359663134997703,-0.039641997689729334,1.0,0.7580165782810444
tamano_familia,0.016638989282745233,0.06599690843099823,-0.25845640944373865,0.890711671869885,0.7831107750000581,0.29644532097653414,-0.009174914777952252,-0.04904337230176844,0.7580165782810444,1.0


Correlación de Spearman:


,Survived,Pclass,Age,SibSp,Parch,Fare,TieneCabina,EdadFaltante,pasajeros_tiquete,tamano_familia
Survived,1.0,-0.3396679366500525,-0.038568119357772046,0.08887948468090501,0.13826563286545587,0.30581904474628074,0.316911523112296,-0.09219652324217223,0.18652946148913782,0.16546291168480365
Pclass,-0.3396679366500525,1.0,-0.4148606254568256,-0.04301876651204207,-0.022801341928590464,-0.7296406038722933,-0.6832907054005039,0.18915633209332727,-0.17426938585724724,-0.0756977911251503
Age,-0.038568119357772046,-0.4148606254568256,1.0,-0.16617897874193427,-0.23547474300564822,0.20155799658124024,0.27132492253666124,-0.07711093855519473,-0.1564063157692528,-0.20409768675917567
SibSp,0.08887948468090501,-0.04301876651204207,-0.16617897874193427,1.0,0.45001397100861634,0.4348477249181958,0.0524210045380025,-0.08773401125023192,0.5709593147435339,0.8491645107690946
Parch,0.13826563286545587,-0.022801341928590464,-0.23547474300564822,0.45001397100861634,1.0,0.3995078224628378,0.08035259378058385,-0.1422624195066149,0.5944533329776867,0.8012240815976783
Fare,0.30581904474628074,-0.7296406038722933,0.20155799658124024,0.4348477249181958,0.3995078224628378,1.0,0.5561626040124004,-0.1605282872259947,0.6519051789020704,0.5143784498226744
TieneCabina,0.316911523112296,-0.6832907054005039,0.27132492253666124,0.0524210045380025,0.08035259378058385,0.5561626040124004,1.0,-0.14411087922884733,0.16937659739968963,0.10929552763957309
EdadFaltante,-0.09219652324217223,0.18915633209332727,-0.07711093855519473,-0.08773401125023192,-0.1422624195066149,-0.1605282872259947,-0.14411087922884733,1.0,-0.09686429747637895,-0.1446020139861164
pasajeros_tiquete,0.18652946148913782,-0.17426938585724724,-0.1564063157692528,0.5709593147435339,0.5944533329776867,0.6519051789020704,0.16937659739968963,-0.09686429747637895,1.0,0.6736039640418285
tamano_familia,0.16546291168480365,-0.0756977911251503,-0.20409768675917567,0.8491645107690946,0.8012240815976783,0.5143784498226744,0.10929552763957309,-0.1446020139861164,0.6736039640418285,1.0


In [40]:
# Pares de variables con mayor correlación absoluta (Spearman), sin repetir pares
pares = corr_spearman.abs().where(np.triu(np.ones(corr_spearman.shape, dtype=bool), k=1)).stack()
pares.index.names = ["Variable 1", "Variable 2"]
pares.sort_values(ascending=False).rename("|rho| Spearman").head(10).to_frame()

|rho| Spearman
Variable 1        Variable 2                          
SibSp             tamano_familia    0.8491645107690946
Parch             tamano_familia    0.8012240815976783
Pclass            Fare              0.7296406038722933
                  TieneCabina       0.6832907054005039
pasajeros_tiquete tamano_familia    0.6736039640418285
Fare              pasajeros_tiquete 0.6519051789020704
Parch             pasajeros_tiquete 0.5944533329776867
SibSp             pasajeros_tiquete 0.5709593147435339
Fare              TieneCabina       0.5561626040124004
                  tamano_familia    0.5143784498226744

#### 5.5.2 Riesgo de data leakage

Hay *data leakage* cuando una variable contiene información que no estaría disponible al momento de predecir o que se construyó a partir del resultado (`Survived`). El diagnóstico mostró que los faltantes de `Cabin` y `Age` dependen de `Survived`, así que se revisa la tasa de supervivencia según los indicadores creados en 5.3 y la correlación de cada variable con `Survived`.

In [41]:
# Tasa de supervivencia (%) según los indicadores de faltantes
for indicador in ["TieneCabina", "EdadFaltante"]:
    tabla = df_limpio.groupby(indicador)["Survived"].agg(["count", "mean"])
    tabla["mean"] = 100 * tabla["mean"]
    tabla.columns = ["Pasajeros", "% sobrevivió"]
    print(f"Supervivencia según {indicador}:")
    display(tabla)

# Correlación de cada variable con el objetivo
corr_spearman["Survived"].drop("Survived").sort_values(key=abs, ascending=False).rename("rho con Survived").to_frame()

Supervivencia según TieneCabina:


,Pasajeros,% sobrevivió
TieneCabina,,
0,687,29.985443959243085
1,204,66.66666666666666


Supervivencia según EdadFaltante:


,Pasajeros,% sobrevivió
EdadFaltante,,
0,714,40.61624649859944
1,177,29.37853107344633


,rho con Survived
Pclass,-0.3396679366500525
TieneCabina,0.316911523112296
Fare,0.30581904474628074
pasajeros_tiquete,0.18652946148913782
tamano_familia,0.16546291168480365
Parch,0.13826563286545587
EdadFaltante,-0.09219652324217223
SibSp,0.08887948468090501
Age,-0.038568119357772046


**Decisión de columnas:**

| Columna | Decisión | Justificación |
|---|---|---|
| `PassengerId` | **Conservar** como clave | Identificador sin valor predictivo (no se usa en el modelo), pero se mantiene en el CSV para rastrear cada fila, como propuso el diagnóstico. |
| `Name` | **Eliminar** con `df.drop`, después de extraer `Titulo` | Identificador (un valor por fila). El título (Mr, Mrs, Miss…) sí resume información de sexo, edad y estado civil, como propuso el diagnóstico. |
| `Ticket` | **Eliminar** con `df.drop` | Casi identificador (681 valores distintos) y con los faltantes `"LINE"`. Su información útil, el tamaño del grupo, ya quedó en `pasajeros_tiquete` (5.4.3). |
| `SibSp`, `Parch` | **Eliminar** con `df.drop` | `tamano_familia` es su suma exacta: conservar las tres crea colinealidad perfecta, y la correlación de cada una con `tamano_familia` es alta (ver tablas). |
| `tamano_familia` | **Conservar** | Resume `SibSp` y `Parch` en una sola variable. |
| `pasajeros_tiquete` | **Conservar** | Está correlacionada con `tamano_familia` pero no la duplica (ver tabla de pares): un tiquete puede agrupar a personas que no son familia. Permite interpretar `Fare` como precio de grupo. |
| `Pclass`, `Fare` | **Conservar** ambas | Están correlacionadas, pero no miden lo mismo (clase del tiquete frente a precio pagado), como concluyó el diagnóstico. |
| `Survived` | **Conservar** | Variable objetivo. |

**Columnas con riesgo de data leakage:**

- **`TieneCabina`:** la tasa de supervivencia cambia mucho según tenga o no cabina registrada (tabla de arriba). Si el registro de la cabina se completó después del naufragio (por ejemplo, con información aportada por sobrevivientes), el indicador estaría codificando parte del resultado. Se conserva, marcada como riesgo.
- **`EdadFaltante`:** el mismo argumento aplica a la edad (más faltantes entre quienes no sobrevivieron, 4.3.1). Se conserva, marcada como riesgo.
- **Valores calculados con todo el dataset (`Age`, `Fare`, `pasajeros_tiquete`):** las medianas, los percentiles y los conteos por tiquete se calcularon con las 891 filas. Si después se separa en entrenamiento y prueba, deben recalcularse solo con el conjunto de entrenamiento para no filtrar información del conjunto de prueba.

In [42]:
# Extraer el título del nombre antes de eliminar Name ("Apellido, Título. Nombres")
df_limpio["Titulo"] = df_limpio["Name"].str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
print("Títulos sin extraer:", int(df_limpio["Titulo"].isna().sum()))
print(df_limpio["Titulo"].value_counts())

Títulos sin extraer: 0
Titulo
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64


In [43]:
# Eliminación de identificadores y columnas redundantes con df.drop
COLS_ELIMINAR = ["Name", "Ticket", "SibSp", "Parch"]

shape_antes = df_limpio.shape
df_limpio = df_limpio.drop(columns=COLS_ELIMINAR)
print("Columnas eliminadas con drop:", COLS_ELIMINAR)
print("shape antes de drop:  ", shape_antes)
print("shape después de drop:", df_limpio.shape)
print("\nColumnas que quedan:", list(df_limpio.columns))

Columnas eliminadas con drop: ['Name', 'Ticket', 'SibSp', 'Parch']
shape antes de drop:   (891, 16)
shape después de drop: (891, 12)

Columnas que quedan: ['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'TieneCabina', 'EdadFaltante', 'pasajeros_tiquete', 'tamano_familia', 'Titulo']


In [44]:
# Codificación de las variables de texto como categóricas (propuesta del diagnóstico para Sex)
COLS_CATEGORICAS = ["Sex", "Embarked", "Titulo"]
df_limpio[COLS_CATEGORICAS] = df_limpio[COLS_CATEGORICAS].astype("category")
for col in COLS_CATEGORICAS:
    print(f"{col}:", list(df_limpio[col].cat.categories))

registro_pasos.append(("5.5 tamano_familia + Titulo + drop + category", df_limpio.shape))

Sex: ['female', 'male']
Embarked: ['C', 'Q', 'S']
Titulo: ['Capt', 'Col', 'Don', 'Dr', 'Jonkheer', 'Lady', 'Major', 'Master', 'Miss', 'Mlle', 'Mme', 'Mr', 'Mrs', 'Ms', 'Rev', 'Sir', 'the Countess']


### 5.6 Exportación (punto 11)

Se exporta el dataset limpio a `titanic_limpio.csv` con `index=False`, se imprime el shape final y los dtypes, y se vuelve a leer el archivo para comprobar que quedó igual a `df_limpio`. Al leerlo, las columnas categóricas se indican explícitamente, porque el CSV no guarda el tipo `category`.

In [45]:
RUTA_LIMPIO = Path("titanic_limpio.csv")
df_limpio.to_csv(RUTA_LIMPIO, index=False)

print("shape final:", df_limpio.shape)
print("\ndtypes:")
print(df_limpio.dtypes)

# Verificación: el archivo exportado coincide con df_limpio
df_verificacion = pd.read_csv(RUTA_LIMPIO, dtype={col: "category" for col in COLS_CATEGORICAS})
pd.testing.assert_frame_equal(df_verificacion, df_limpio.reset_index(drop=True))
print("\nArchivo exportado y verificado:", RUTA_LIMPIO)
print("Nulos en el archivo exportado:", int(df_verificacion.isna().sum().sum()))

shape final: (891, 12)

dtypes:
PassengerId             int64
Survived                int64
Pclass                  int64
Sex                  category
Age                   float64
Fare                  float64
Embarked             category
TieneCabina             int64
EdadFaltante            int64
pasajeros_tiquete       int64
tamano_familia          int64
Titulo               category
dtype: object

Archivo exportado y verificado: titanic_limpio.csv
Nulos en el archivo exportado: 0


### 5.7 Resumen de la limpieza

Se reúne lo eliminado, lo imputado y lo agregado, calculado a partir de los objetos del notebook.

In [46]:
# Evolución del shape en cada paso
display(pd.DataFrame(registro_pasos, columns=["Paso", "shape"]).set_index("Paso"))

# Columnas eliminadas y agregadas respecto al original
print("Columnas eliminadas:", list(df.columns.difference(df_limpio.columns)))
print("Columnas agregadas: ", list(df_limpio.columns.difference(df.columns)))

# Valores imputados y acotados por columna
resumen_tratamiento = pd.DataFrame({
    "Valores modificados": [len(filas_embarked_nulo), len(filas_fare_nulo), int(df_limpio["EdadFaltante"].sum()), int(fare_modificada.sum())],
    "Tratamiento": ["Imputación con la moda", "Imputación con la mediana por Pclass", "Imputación con la mediana por Pclass y Sex", "Winsorización p5-p95"],
}, index=["Embarked", "Fare (ceros)", "Age", "Fare (colas)"])
display(resumen_tratamiento)

print("Filas eliminadas:", df.shape[0] - df_limpio.shape[0])
print("shape original:", df.shape, "| shape final:", df_limpio.shape)

,shape
Paso,
Original,"(891, 12)"
5.1 drop_duplicates,"(891, 12)"
5.2 Boolean Indexing,"(891, 12)"
5.3.1 dropna (Cabin) + TieneCabina,"(891, 12)"
"5.3 fillna (Embarked, Fare, Age) + EdadFaltante","(891, 13)"
5.4 winsorización Fare + pasajeros_tiquete,"(891, 14)"
5.5 tamano_familia + Titulo + drop + category,"(891, 12)"


Columnas eliminadas: ['Cabin', 'Name', 'Parch', 'SibSp', 'Ticket']
Columnas agregadas:  ['EdadFaltante', 'TieneCabina', 'Titulo', 'pasajeros_tiquete', 'tamano_familia']


,Valores modificados,Tratamiento
Embarked,2,Imputación con la moda
Fare (ceros),15,Imputación con la mediana por Pclass
Age,177,Imputación con la mediana por Pclass y Sex
Fare (colas),85,Winsorización p5-p95


Filas eliminadas: 0
shape original: (891, 12) | shape final: (891, 12)


### 5.8 Análisis y conclusiones del Ejercicio 2

La limpieza siguió, paso a paso, lo que encontró el diagnóstico (tabla de trazabilidad al inicio del ejercicio). En resumen:

**Qué se eliminó**

- **Filas:** ninguna. No hubo duplicados (ni exactos ni ignorando `PassengerId`), ninguna fila incumplió las reglas de dominio, y los registros atípicos del diagnóstico se conservaron a propósito: las 82 filas con 2 o más atributos atípicos siguen en el dataset, porque son pasajeros reales (familias numerosas y pasajeros de 1.ª clase).
- **Columnas (5):**
  - `Cabin`, la columna más problemática (4.4), con `dropna` y un umbral del 50 % de valores no nulos; solo tenía 204 de 891.
  - `Name` y `Ticket`, por ser identificadores o casi identificadores.
  - `SibSp` y `Parch`, por ser redundantes con `tamano_familia`, que es su suma exacta.

**Qué se imputó y qué se acotó**

- **Imputación (194 valores):**
  - `Age`: 177 valores, con la mediana de su grupo `Pclass` × `Sex`, porque su mecanismo es MAR.
  - `Fare`: 15 ceros disfrazados, con la mediana de su clase (61.9792, 15.0229 y 8.05).
  - `Embarked`: 2 valores, con la moda (`S`), porque su mecanismo es MCAR.
- **Winsorización (85 valores de `Fare`):** se subieron 40 valores al percentil 5 (7.2292) y se bajaron 45 al percentil 95 (112.07915).

**Qué se agregó (5 columnas):**
- `TieneCabina` y `EdadFaltante`: conservan la señal de los faltantes MAR.
- `pasajeros_tiquete`: conserva el hallazgo de la tarifa de grupo.
- `tamano_familia`: resume `SibSp` y `Parch`.
- `Titulo`: resume `Name`.

`Sex`, `Embarked` y `Titulo` quedaron como categóricas.

**Resultado:** se pasa de **891 filas × 12 columnas** a **891 filas × 12 columnas** (5 columnas salen y 5 entran), sin ningún nulo, exportado en `titanic_limpio.csv`.

**Efecto de las transformaciones**

- **Imputación de `Age`:**
  - La media baja de 29.69911764705882 a 29.11242424242424 años y la mediana de 28.0 a 26.0.
  - El cambio más grande está en la dispersión: la desviación estándar cae de 14.526497332334044 a 13.304424074688153, porque 177 pasajeros quedan concentrados en solo seis valores (las medianas de grupo). En la figura se ve como un pico artificial entre 25 y 30 años.
  - Imputar por grupo respeta el mecanismo MAR que encontró el diagnóstico, pero no recupera la variabilidad real; por eso se conservó `EdadFaltante`.
- **Winsorización de `Fare`:**
  - El diagnóstico propuso conservar estas tarifas o transformarlas, y la winsorización hace eso: no elimina filas y conserva el orden de las tarifas (Spearman de 0.9998221135170301 entre la serie antes y después).
  - La media pasa de 32.204207968574636 a 28.22123552188552, y la desviación estándar de 49.693428597180905 a 29.14891277694091. El máximo pasa de 512.3292 a 112.07915.
  - La mediana casi no cambia (de 14.4542 a 14.5, por la imputación de los ceros, no por el recorte).
  - Tukey habría recortado 116 valores; la winsorización solo modifica 85.
- **Registros atípicos y tarifa de grupo:**
  - La winsorización acotó la tarifa de 25 de las 82 filas con varios atributos atípicos, y de **ninguna** de las 52 tarifas atípicas dentro de 3.ª clase. Esas tarifas son atípicas solo frente a su clase, porque son precios de tiquetes de grupo.
  - Se evaluó corregirlas con una tarifa por persona, pero empeora el problema en 2.ª y 3.ª clase (los atípicos dentro de la clase suben de 7 a 28 y de 52 a 103). Por eso se conservó `Fare` como precio del tiquete y se agregó `pasajeros_tiquete` para que el modelo pueda interpretarla.
- **Redundancia:**
  - `tamano_familia` reemplaza a `SibSp` y `Parch` (|rho| de Spearman de 0.8491645107690946 y 0.8012240815976783 con ellas).
  - `pasajeros_tiquete` está correlacionada con `tamano_familia` (|rho| = 0.6736039640418285) pero no la duplica, y se relaciona más con `Survived` que ella (0.18652946148913782 frente a 0.16546291168480365).
  - `Pclass` y `Fare` siguen muy correlacionadas (|rho| = 0.7296406038722933), igual que `TieneCabina` con `Pclass` (|rho| = 0.6832907054005039). Se conservan, pero un modelo lineal sufriría colinealidad entre ellas.

**Riesgos para el modelo predictivo**

- **Data leakage:**
  - `TieneCabina` muestra una diferencia de supervivencia muy grande: 66.66666666666666 % con cabina frente a 29.985443959243085 % sin cabina.
  - `EdadFaltante` muestra una menor: 29.37853107344633 % con edad faltante frente a 40.61624649859944 % con edad registrada.
  - Si esos faltantes se generaron después del naufragio, los indicadores filtran parte del resultado. Conviene entrenar el modelo con y sin ellos y comparar.
- **Estadísticos calculados con todo el dataset:** las medianas de imputación, los percentiles 5 y 95 y los conteos de `pasajeros_tiquete` se calcularon con las 891 filas. Para modelar, deben recalcularse solo con el conjunto de entrenamiento.
- **`Titulo`:** tiene categorías con 1 o 2 pasajeros (`Don`, `Lady`, `Sir`, `Capt`, etc.). Antes de modelar conviene agruparlas en una categoría `Otro`.